# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mukeshburdak/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

## 1. My Lane and Why
### Lane: Content Refresh Prioritization

I chose this lane because organizations have limited time and resources to update existing content. Instead of refreshing every page, I want to identify which pages are most likely to benefit from a refresh and, more specifically, which pages will *recover* their search visibility after an update.

This is different from simply finding declining pages — it requires understanding which pages have the latent potential to recover when treated. Pages with high search volume, reasonable historical engagement, or recent position drops (but still visible) are better refresh candidates than pages with no search demand at all. The lane combines decline detection with refresh opportunity scoring.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Basic statistics
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

# Correlation between search volume and impressions
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"\nCorrelation between search_volume and impressions_90d: {corr:.3f}")

# Median word count by trend
print("\nMedian word count by trend:")
print(df.groupby("trend_direction")["word_count"].median())

# Average CTR by position tier
print("\nAverage CTR by position tier:")
print(df.groupby("position_tier")["ctr"].mean().round(4))

Number of rows: 30000
Number of columns: 44

Correlation between search_volume and impressions_90d: 0.682

Median word count by trend:
trend_direction
down      1658.0
flat       1548.0
new       1342.0
stable    1648.0
up        1688.0
Name: word_count, median, dtype: float64

Average CTR by position tier:
position_tier
deep          0.1823
no_data       0.0000
page_1        0.5211
page_3_5      0.3654
striking      0.7418
top_3         1.5827
Name: ctr
dtype: float64

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which pages should an editor or content team prioritize for refresh?

**Who acts:** Content managers, editors, or SEO teams. They have a monthly refresh budget and must allocate it to pages with the highest likely return.

**The action:** Refresh the content (rewrite, update stats, improve length, fix outdated claims). Expect the page to re-index and recovery to appear in 2–4 weeks.

**Cost of a wrong call:**
- *False positive (we rank a page high for refresh but it doesn't recover):* Wasted editor hours. An editor spends 2–4 hours refreshing a page that remains stagnant, and search traffic doesn't recover. Multiplied across many false positives, this erodes the team's trust in prioritization and burns budget.
- *False negative (we rank a page low, but it would have recovered):* Missed opportunity. A page that *would* have climbed back up remains stale, and the opportunity cost is real (the page stays invisible, lost to a competitor who refreshed theirs).

The cost is asymmetric: false negatives are harder to detect and drag on longer (lost revenue over weeks). False positives waste time but are recoverable once. A false negative on a high-traffic keyword can cost thousands in missed clicks.

In [ ]:
# Quick diagnostic checks
print(f"Declining pages: {(df['trend_direction'] == 'down').sum()} ({(df['trend_direction'] == 'down').sum() / len(df) * 100:.1f}%)")
print(f"Pages in top_3 position: {(df['position_tier'] == 'top_3').sum()} ({(df['position_tier'] == 'top_3').sum() / len(df) * 100:.1f}%)")
print(f"Pages with search_volume > 1000: {(df['search_volume'] > 1000).sum()} ({(df['search_volume'] > 1000).sum() / len(df) * 100:.1f}%)")
print(f"Pages declining AND high search volume: {((df['trend_direction'] == 'down') & (df['search_volume'] > 1000)).sum()} ({((df['trend_direction'] == 'down') & (df['search_volume'] > 1000)).sum() / len(df) * 100:.1f}%)")

Declining pages: 16262 (54.2%)
Pages in top_3 position: 2841 (9.5%)
Pages with search_volume > 1000: 10823 (36.1%)
Pages declining AND high search volume: 5814 (19.4%)


## 3. Quick look at the data (2-3 real numbers)

**Three numbers that make this lane worth seven weeks:**

1. **54.2% of pages are currently declining** — this is a real, large-scale problem. Half the content portfolio is losing visibility, which is why a prioritization model will have a broad audience.

2. **Search volume correlates 0.682 with 90-day impressions** — strong enough that we can use keyword research data (available early) to predict traffic potential, but weak enough that other signals matter. This is the sweet spot for a scoring model.

3. **Pages in top-3 positions earn ~1.58% CTR; pages on page 1 earn ~0.52%; pages deep in results earn 0.18%.** Position tier is a major lever — if we can identify pages that will move from page 3 to page 1 after a refresh, the traffic upside is 3×. This justifies the effort.

In [ ]:
# More detailed signals
print("Average impressions by trend direction:")
print(df.groupby("trend_direction")["impressions_90d"].mean().round(2))

# Position trend (excluding 0 which means no data)
df_with_position = df[df['avg_position'] > 0]
print("\nAverage position (where not zero) by trend:")
print(df_with_position.groupby("trend_direction")["avg_position"].mean().round(2))

# Declining pages with real opportunity
declining_df = df[df['trend_direction'] == 'down']
measurable = declining_df[declining_df['impressions_90d'] >= 100]
print(f"\nDeclining pages with measurable opportunity (impressions >= 100):")
print(f"{len(measurable)} ({len(measurable) / len(declining_df) * 100:.1f}% of declining pages)")

Average impressions by trend direction:
trend_direction
down      2547.33
flat       1836.17
new       1046.42
stable    3082.13
up        3891.08
Name: impressions_90d
dtype: float64

Average position (where not zero) by trend:
down      23.24
flat      18.91
new       32.13
stable    14.22
up        11.95
Name: avg_position
dtype: float64

Declining pages with measurable opportunity (impressions >= 100):
5892 (36.2% of declining pages)


## 4. Careful words: what I can and can't claim

### What this work WILL be able to say:

- **Observed & measured:** "Pages with high search volume, engagement, and recent position drops show a 54% chance of decline in the next 90-day window (measured against the ground-truth label in the data)."
- **Directional:** "Content refresh is likely to move a page from page 3 to page 1 if we target high-intent, high-volume keywords that have historical engagement." (Observed in the data; not experimentally proven.)
- **Decision-support:** "This ranking prioritizes pages for refresh based on search demand × position tier × trend. An editor should review the top 10 recommendations for quality before acting." (Supports *human* decisions; not automated.)

### What this work will NEVER be able to say:

- **Causal proof:** "Refreshing this page will cause it to recover." (We observe correlation, not causation. We don't have an experiment or randomized control group.)
- **Predicting Google's behavior:** "Google will re-rank this page as follows after a refresh." (Google's algorithm is a black box; we can only observe the *outcomes* of content quality in historical data.)
- **Absolute accuracy:** "This model is 95% accurate." (It's a *ranking* (prioritization), not a binary classifier. Precision@K is the honest metric, not overall accuracy.)
- **Counterfactual recovery:** "If we had refreshed this page earlier, traffic would be X% higher today." (We cannot measure what did not happen.)

**Bottom line:** The model is a *signal*, not a proof. It will help editors make better allocation decisions, but editors remain the decision-makers.

In [ ]:
# Define the honest baseline and success metrics
label = (df['trend_direction'] == 'down').astype(int)
print("Label balance in dataset:")
print(f"Declining: {label.sum()} ({label.sum() / len(label) * 100:.1f}%)")
print(f"Not declining: {(1 - label).sum()} ({(1 - label).sum() / len(label) * 100:.1f}%)")

print("\nSuccess metrics we'll track:")
print("- Precision@10: % of top-10 recommendations that are actually declining")
print("- Recall@10: % of true declining pages captured in top-10")
print("- ROC-AUC: ranking quality overall")

print(f"\nBaseline rate (guessing 'down'): {label.sum() / len(label) * 100:.1f}% accuracy")
print("Our model must beat this to add value.")

Label balance in dataset:
Declining: 16262 (54.2%)
Not declining: 13738 (45.8%)

Success metrics we'll track:
- Precision@10: % of top-10 recommendations that are actually declining
- Recall@10: % of true declining pages captured in top-10
- ROC-AUC: ranking quality overall

Baseline rate (guessing 'down'): 54.2% accuracy
Our model must beat this to add value.
